In [4]:
import subprocess
import os
import random
import shutil
from pathlib import Path

In [5]:
HIDF_DATASET_PATH = "data/hidf_subset"
CIFAKE_DATASET_PATH = "data/cifake_subset"
SID_DATASET_PATH = "data/sid_subset"

HIDF_140K_DATASET_PATH = "data/hidf_140k_subset"
HIDF_140K_SOURCE_PATH = "140k_reduced"

In [9]:
def run_command(cmd):
    process = subprocess.Popen(
        cmd,
        shell=True,
        cwd='/root/Deepfake_Detection',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )
    if process.stdout is not None:
        for line in process.stdout:
            print(line, end="")
    process.wait()
    return process.returncode == 0


MAX_SAMPLES_PER_CLASS = 10
SEED = 42


def sample_files(src_dir: str, dst_dir: str, fraction: float, seed: int = 42):
    if not os.path.isdir(src_dir):
        return
    os.makedirs(dst_dir, exist_ok=True)
    files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]
    if not files:
        return
    rng = random.Random(seed + hash(src_dir) % 2**32)
    rng.shuffle(files)
    n = min(MAX_SAMPLES_PER_CLASS, max(1, int(round(len(files) * fraction))))
    for fname in files[:n]:
        shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))


def sample_and_split(src_dir: str, train_dir: str, test_dir: str, total_fraction: float, test_ratio: float, seed: int = 42):
    if not os.path.isdir(src_dir):
        return
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]
    if not files:
        return
    rng = random.Random(seed + hash(src_dir) % 2**32)
    rng.shuffle(files)
    total_n = min(MAX_SAMPLES_PER_CLASS, max(2, int(round(len(files) * total_fraction))))
    test_n = max(1, int(round(total_n * test_ratio)))
    train_n = max(1, total_n - test_n)
    for fname in files[:train_n]:
        shutil.copy2(os.path.join(src_dir, fname), os.path.join(train_dir, fname))
    for fname in files[train_n: train_n + test_n]:
        shutil.copy2(os.path.join(src_dir, fname), os.path.join(test_dir, fname))


def prepare_dataset_subset(name: str, src_root: str, dst_root: str):
    src_root = Path(src_root)
    dst_root = Path(dst_root)
    if name == "sid":
        sample_and_split(
            str(src_root / "real"),
            str(dst_root / "train" / "real"),
            str(dst_root / "test" / "real"),
            COPY_FRACTION,
            1 / 3,
            SEED,
        )
        sample_and_split(
            str(src_root / "synthetic"),
            str(dst_root / "train" / "fake"),
            str(dst_root / "test" / "fake"),
            COPY_FRACTION,
            1 / 3,
            SEED,
        )
        sample_and_split(
            str(src_root / "tampered"),
            str(dst_root / "train" / "fake"),
            str(dst_root / "test" / "fake"),
            COPY_FRACTION,
            1 / 3,
            SEED,
        )
    else:
        sample_files(
            str(src_root / "train" / "fake"),
            str(dst_root / "train" / "fake"),
            COPY_FRACTION,
            SEED,
        )
        sample_files(
            str(src_root / "train" / "real"),
            str(dst_root / "train" / "real"),
            COPY_FRACTION,
            SEED,
        )
        sample_files(
            str(src_root / "test" / "fake"),
            str(dst_root / "test" / "fake"),
            COPY_FRACTION,
            SEED,
        )
        sample_files(
            str(src_root / "test" / "real"),
            str(dst_root / "test" / "real"),
            COPY_FRACTION,
            SEED,
        )

In [5]:
# Evaluate SID model on SID subset test
print("Evaluating SID model on SID subset test...")
cmd = f"python inference.py --checkpoint checkpoints/sid/sid_trained.pt --data-root data/sid_subset --batch-size 4 "
run_command(cmd)

Evaluating SID model on SID subset test...
CUDA available: True
Test classes: ['fake', 'real']

Test loss: 0.4818

Test metrics
  Accuracy:  0.7500
  Precision: 0.7500
  Recall:    1.0000
  F1-score:  0.8571
  Confusion matrix (rows=true, cols=pred):
            fake      real
    fake       300         0
    real       100         0


True

In [6]:
# Evaluate CiFake model on CiFake test
print("Evaluating CiFake model on CiFake test...")
cmd = f'python inference.py --checkpoint checkpoints/cifake/cifake_trained.pt --data-root "{CIFAKE_DATASET_PATH}" --batch-size 4 '
run_command(cmd)

Evaluating CiFake model on CiFake test...
CUDA available: True
Test classes: ['fake', 'real']

Test loss: 0.3702

Test metrics
  Accuracy:  0.8334
  Precision: 0.8200
  Recall:    0.8556
  F1-score:  0.8374
  Confusion matrix (rows=true, cols=pred):
            fake      real
    fake      1298       219
    real       285      1223


True

In [7]:
# Evaluate HIDF model on HIDF test
print("Evaluating HIDF model on HIDF test...")
cmd = f"python inference.py --checkpoint checkpoints/hidf/hidf_trained.pt --data-root {HIDF_DATASET_PATH} --batch-size 4 "
run_command(cmd)

Evaluating HIDF model on HIDF test...
CUDA available: True
Test classes: ['fake', 'real']

Test loss: 0.6942

Test metrics
  Accuracy:  0.5197
  Precision: 0.5024
  Recall:    0.7704
  F1-score:  0.6082
  Confusion matrix (rows=true, cols=pred):
            fake      real
    fake       104        31
    real       103        41


True

In [ ]:
# Finetune HIDF on 140k Dataset
print("Preparing HIDF finetune subset from 140k_reduced...")
prepare_dataset_subset("140k_reduced", HIDF_140K_SOURCE_PATH, HIDF_140K_DATASET_PATH)
print("Finetuning Hidf model on 140k subset...")
cmd = f"python train.py --data-root {HIDF_140K_DATASET_PATH} --experiment-name hidf --epochs 10 --batch-size 4 --val-fraction 0.333 --safe-cuda"

run_command(cmd)
shutil.copy2("checkpoints/hidf/best.pt", "checkpoints/hidf/finetuned_hidf.pt")

In [10]:
import torch

# Make paths robust no matter where the notebook kernel starts
repo_root_candidates = [Path.cwd(), Path.cwd().parent, Path("/root/Deepfake_Detection")]
REPO_ROOT = None
for cand in repo_root_candidates:
    if (cand / "inference.py").exists() and (cand / "checkpoints").exists():
        REPO_ROOT = cand
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root. Please run from /root/Deepfake_Detection.")


def resolve_checkpoint(exp_name: str, preferred_name: str) -> Path:
    exp_dir = REPO_ROOT / "checkpoints" / exp_name
    if not exp_dir.exists():
        raise FileNotFoundError(f"Missing checkpoint directory: {exp_dir}")

    preferred = exp_dir / preferred_name
    if preferred.exists():
        return preferred

    trained = sorted(exp_dir.glob("*_trained.pt"))
    if trained:
        return trained[0]

    best = exp_dir / "best.pt"
    if best.exists():
        return best

    all_pts = sorted(exp_dir.glob("*.pt"))
    if all_pts:
        return all_pts[0]

    raise FileNotFoundError(f"No .pt checkpoint found in {exp_dir}")


# Build one weighted-average checkpoint from three trained models
CHECKPOINTS_TO_MERGE = [
    resolve_checkpoint("sid", "sid_trained.pt"),
    resolve_checkpoint("hidf", "hidf_trained.pt"),
    resolve_checkpoint("cifake", "cifake_trained.pt"),
]
CHECKPOINT_WEIGHTS = [1 / 3, 1 / 3, 1 / 3]  # weighted average

ensemble_dir = REPO_ROOT / "checkpoints" / "ensemble"
ensemble_dir.mkdir(parents=True, exist_ok=True)
combined_ckpt_path = ensemble_dir / "weighted_avg_sid_hidf_cifake.pt"


def load_checkpoint_raw(path: Path):
    return torch.load(path, map_location="cpu")


def weighted_average_state_dict(state_dicts, weights):
    if len(state_dicts) != len(weights):
        raise ValueError("state_dict count and weight count must match")

    merged = {}
    keys = state_dicts[0].keys()
    for key in keys:
        tensors = [sd[key] for sd in state_dicts]
        first = tensors[0]

        # Average only floating tensors; keep non-float tensors from first model
        if torch.is_tensor(first) and torch.is_floating_point(first):
            out = torch.zeros_like(first, dtype=torch.float32)
            for tensor, w in zip(tensors, weights):
                out += tensor.float() * w
            merged[key] = out.to(dtype=first.dtype)
        else:
            merged[key] = first
    return merged


raw_ckpts = [load_checkpoint_raw(p) for p in CHECKPOINTS_TO_MERGE]
state_dicts = [ckpt["model_state_dict"] for ckpt in raw_ckpts]
merged_state = weighted_average_state_dict(state_dicts, CHECKPOINT_WEIGHTS)

combined_payload = dict(raw_ckpts[0])
combined_payload["model_state_dict"] = merged_state
combined_payload["merged_from"] = [str(p) for p in CHECKPOINTS_TO_MERGE]
combined_payload["merge_weights"] = CHECKPOINT_WEIGHTS

torch.save(combined_payload, combined_ckpt_path)
print(f"Saved combined model checkpoint to: {combined_ckpt_path}")


# Create a tiny test-only subset from each dataset (2 fake + 2 real images)
def copy_tiny_test_subset(src_root: str, dst_root: str, max_per_class: int = 2, seed: int = 42):
    src_root = Path(src_root)
    dst_root = Path(dst_root)
    image_exts = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp"}

    for cls_name in ["fake", "real"]:
        src_cls = src_root / "test" / cls_name
        dst_cls = dst_root / "test" / cls_name
        if not src_cls.exists():
            print(f"Skipping missing class folder: {src_cls}")
            continue

        dst_cls.mkdir(parents=True, exist_ok=True)
        files = [p for p in src_cls.rglob("*") if p.is_file() and p.suffix.lower() in image_exts]
        if not files:
            print(f"No valid image files found in: {src_cls}")
            continue

        rng = random.Random(seed + hash(str(src_cls)) % (2**32))
        rng.shuffle(files)
        for i, f in enumerate(files[:max_per_class]):
            # Prefix copied name to avoid collisions from nested folders
            dst_name = f"{i}_{f.name}"
            shutil.copy2(f, dst_cls / dst_name)


# Keep this cell runnable by itself
LOCAL_SEED = 42

TINY_ROOT = REPO_ROOT / "data" / "tiny_eval"
if TINY_ROOT.exists():
    shutil.rmtree(TINY_ROOT)

DATASETS = {
    "sid": REPO_ROOT / SID_DATASET_PATH,
    "cifake": REPO_ROOT / CIFAKE_DATASET_PATH,
    "hidf": REPO_ROOT / HIDF_DATASET_PATH,
}

for name, src in DATASETS.items():
    tiny_dst = TINY_ROOT / name
    copy_tiny_test_subset(src, tiny_dst, max_per_class=2, seed=LOCAL_SEED)


# Evaluate the single combined model on tiny subsets for all three datasets
for name in ["sid", "cifake", "hidf"]:
    data_root = TINY_ROOT / name
    print(f"\nEvaluating combined model on tiny {name.upper()} test set...")
    inference_script = REPO_ROOT / "inference.py"
    cmd = (
        f'python "{inference_script}" --checkpoint "{combined_ckpt_path}" '
        f'--data-root "{data_root}" --batch-size 4'
    )
    run_command(cmd)


Saved combined model checkpoint to: /root/Deepfake_Detection/checkpoints/ensemble/weighted_avg_sid_hidf_cifake.pt

Evaluating combined model on tiny SID test set...
CUDA available: True
Test classes: ['fake', 'real']

Test loss: 0.6549

Test metrics
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1-score:  1.0000
  Confusion matrix (rows=true, cols=pred):
            fake      real
    fake         2         0
    real         0         2

Evaluating combined model on tiny CIFAKE test set...
CUDA available: True
Test classes: ['fake', 'real']

Test loss: 0.7254

Test metrics
  Accuracy:  0.5000
  Precision: 0.0000
  Recall:    0.0000
  F1-score:  0.0000
  Confusion matrix (rows=true, cols=pred):
            fake      real
    fake         0         2
    real         0         2

Evaluating combined model on tiny HIDF test set...
CUDA available: True
Test classes: ['fake', 'real']

Test loss: 0.6936

Test metrics
  Accuracy:  0.5000
  Precision: 0.0000
  Recall:    0.000